In [1]:
%pip install smartnoise-synth

  Using cached smartnoise_synth-1.0.3-py3-none-any.whl.metadata (2.7 kB)
  Using cached Faker-15.3.4-py3-none-any.whl.metadata (15 kB)
  Using cached opacus-0.14.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached pac_synth-0.0.8-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.4 kB)
  Using cached torch-1.13.1-cp38-cp38-manylinux1_x86_64.whl.metadata (24 kB)
  Using cached antlr4_python3_runtime-4.9.3-py3-none-any.whl
  Using cached graphviz-0.17-py3-none-any.whl.metadata (8.1 kB)
INFO: pip is looking at multiple versions of smartnoise-sql to determine which version is compatible with other requirements. This could take a while.
  Using cached smartnoise_sql-1.0.3-py3-none-any.whl.metadata (9.5 kB)
  Using cached opendp-0.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached nvidia_cuda_runtime_cu11-11.7.99-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu11-8.5.0.96-2-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvi

In [1]:
# from data_synthesizer.util import  plot_training_loss, ModelType
from data_loader import DataLoader
from data_evaluator import ClassifierType
from data_synthesizer.pipeline import PipelineBuilder


In [2]:
cat_list_adult = ['workclass','education','marital-status','occupation','relationship','race','sex','native-country','income']
num_list_adult = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
adult_qai_columns = ['education','education-num','marital-status','occupation','relationship','race','sex', 'native-country']
adult_risk_column = ['capital-gain','capital-loss','hours-per-week','native-country','income']
df_real_adult_train = DataLoader('../../data/adult_train.csv').get_dataframe(cat_list_adult, str)
df_real_adult_test = DataLoader('../../data/adult_test.csv').get_dataframe(cat_list_adult, str)

In [4]:
import pandas as pd
from snsynth import Synthesizer


synth = Synthesizer.create("dpctgan", epsilon=2, verbose=True, epochs=1500)

In [6]:
synth.fit(data=df_real_adult_train, categorical_columns=cat_list_adult, continuous_columns=num_list_adult, preprocessor_eps=1.0)

Spent 0.9999999999999999 epsilon on preprocessor, leaving 1.0 for training
Epoch 1, Loss G: 0.6661, Loss D: 1.3805
epsilon is 0.1685669340063301, alpha is 63.0
Epoch 2, Loss G: 0.6527, Loss D: 1.3934
epsilon is 0.209558068077265, alpha is 63.0
Epoch 3, Loss G: 0.6460, Loss D: 1.4003
epsilon is 0.25054920214819987, alpha is 63.0
Epoch 4, Loss G: 0.6600, Loss D: 1.3912
epsilon is 0.2915403362191347, alpha is 63.0
Epoch 5, Loss G: 0.6704, Loss D: 1.3845
epsilon is 0.33253147029006963, alpha is 63.0
Epoch 6, Loss G: 0.6643, Loss D: 1.3914
epsilon is 0.37276384089934517, alpha is 59.0
Epoch 7, Loss G: 0.6699, Loss D: 1.3852
epsilon is 0.409651775143029, alpha is 55.0
Epoch 8, Loss G: 0.6607, Loss D: 1.3874
epsilon is 0.44372685687883123, alpha is 51.0
Epoch 9, Loss G: 0.6604, Loss D: 1.3913
epsilon is 0.47558324567259336, alpha is 48.0
Epoch 10, Loss G: 0.6694, Loss D: 1.3905
epsilon is 0.5056409712903367, alpha is 45.0
Epoch 11, Loss G: 0.6672, Loss D: 1.3877
epsilon is 0.534126549635178, 

In [7]:
data_sample = synth.sample(32561)

In [8]:
data_sample.to_csv('../data/adult_dpctgan_training_first_colab_eps2_preps1.csv', sep=',', index=False)

In [3]:
df_real_adult_synth_eps2_preps1 = DataLoader('../../data/adult_dpctgan_training_first_colab_eps2_preps1.csv').get_dataframe(cat_list_adult, str)

In [4]:
pipeline_builder = PipelineBuilder(df_real_adult_train, cat_list_adult, num_list_adult)

pipeline_builder.add_ressemblance_evaluation_task(df_real_adult_test, df_real_adult_synth_eps2_preps1)

classifier_types = [ClassifierType.CART, 
                    ClassifierType.KNN, 
                    ClassifierType.LDA, 
                    ClassifierType.NB, 
                    ClassifierType.LR, 
                    ClassifierType.RANDOM_FOREST,
                    ClassifierType.SVM,
                    ClassifierType.XGBOOST]
pipeline_builder.add_utility_evaluation_task(df_real_adult_train, classifier_types, df_real_adult_synth_eps2_preps1)
pipeline_builder.add_privacy_evaluation_task(df_real_adult_test,adult_qai_columns, adult_risk_column, df_real_adult_synth_eps2_preps1)
pipeline_builder.add_privacy_anonymeter_evaluation_task(df_real_adult_test, df_real_adult_synth_eps2_preps1)
pipeline_builder.build()
results = pipeline_builder.run()

Utility evaluation in progress.
Privacy evaluation in progress.
5000  ------finished------ 10000
10000  ------finished------ 15000
15000  ------finished------ 20000
20000  ------finished------ 25000
25000  ------finished------ 30000
30000  ------finished------ 32561
32561  ------finished------ 32561
5000  ------finished------ 10000
10000  ------finished------ 15000
15000  ------finished------ 20000
20000  ------finished------ 25000
25000  ------finished------ 30000
30000  ------finished------ 32561
32561  ------finished------ 32561
end
Privacy anonymeter evaluation in progress.


Found 201 failed queries out of 500. Check DEBUG messages for more details.
Found 206 failed queries out of 500. Check DEBUG messages for more details.
Found 399 failed queries out of 500. Check DEBUG messages for more details.


end


In [1]:
import pickle
with open('../../data/df_real_adult_synth_eps2_preps1.pkl', 'wb') as file:
    # Serialize the object and write it to the file
    pickle.dump(results, file)

NameError: name 'results' is not defined